In [ ]:
import json
import os
from pathlib import Path
from pymilvus import MilvusClient, DataType
from dotenv import load_dotenv

In [ ]:
BASE_DIR = Path(r"D:\Final_GRAG")
load_dotenv(BASE_DIR / ".env")

ZILLIZ_CLOUD_URI = os.getenv("ZILLIZ_CLOUD_URI")
ZILLIZ_CLOUD_API_KEY = os.getenv("ZILLIZ_CLOUD_API_KEY")

REPORT_UNITS_DIR = BASE_DIR / "metadata" / "report_units"
REPORT_CHUNK_FILES = sorted(REPORT_UNITS_DIR.glob("*/report_chunks.json"))
COLLECTION_NAME = "report_chunks"

In [ ]:
# Connect to Zilliz Cloud
client = MilvusClient(
    uri=ZILLIZ_CLOUD_URI,
    token=ZILLIZ_CLOUD_API_KEY
)

In [ ]:
# Load report_chunks từ tất cả các báo cáo
all_chunks = []
source_counts = {}

for chunk_file in REPORT_CHUNK_FILES:
    report_id = chunk_file.parent.name
    with open(chunk_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    source_counts[report_id] = len(chunks)
    all_chunks.extend(chunks)

print(len(source_counts))
print(len(all_chunks))

In [ ]:
report_chunks_schema = MilvusClient.create_schema(
    auto_id=False,
    enable_dynamic_field=True,
)

report_chunks_schema.add_field(
    field_name="chunk_id",
    datatype=DataType.VARCHAR,
    max_length=128,
    is_primary=True,
)

report_chunks_schema.add_field(
    field_name="dense_embedding",
    datatype=DataType.FLOAT_VECTOR,
    dim=1024,
)
report_chunks_schema.add_field(
    field_name="sparse_embedding",
    datatype=DataType.SPARSE_FLOAT_VECTOR,
)

report_chunks_schema.add_field(
    field_name="content_text",
    datatype=DataType.VARCHAR,
    max_length=16384,
)

report_chunks_schema.add_field(
    field_name="report_id",
    datatype=DataType.VARCHAR,
    max_length=64,
)
report_chunks_schema.add_field(
    field_name="page_numbers",
    datatype=DataType.VARCHAR,
    max_length=256,
)

report_chunks_schema.add_field(
    field_name="chunk_type",
    datatype=DataType.VARCHAR,
    max_length=32,
)
report_chunks_schema.add_field(
    field_name="modality",
    datatype=DataType.VARCHAR,
    max_length=16,
)
report_chunks_schema.add_field(
    field_name="chunk_index",
    datatype=DataType.INT32,
)

In [ ]:
index_params = client.prepare_index_params()

index_params.add_index(field_name="chunk_id")

# Index cho Dense vector
index_params.add_index(
    field_name="dense_embedding",
    index_type="AUTOINDEX",
    metric_type="COSINE",
)

# Index cho Sparse vector
index_params.add_index(
    field_name="sparse_embedding",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="IP",
)

index_params.add_index(field_name="report_id")
index_params.add_index(field_name="chunk_type")
index_params.add_index(field_name="modality")

In [ ]:
# Create collection
if "report_chunks" in client.list_collections():
    client.drop_collection("report_chunks")

client.create_collection(
    collection_name="report_chunks",
    schema=report_chunks_schema,
    index_params=index_params,
)

In [ ]:
prepared_chunks = []

for chunk in all_chunks:
    prepared = chunk.copy()
    
    # Convert sparse embedding format: {str_key: val} → {int_key: float_val}
    if "sparse_embedding" in prepared:
        sparse = prepared["sparse_embedding"]
        if isinstance(sparse, dict):
            if "indices" in sparse and "values" in sparse:
                prepared["sparse_embedding"] = dict(zip(sparse["indices"], sparse["values"]))
            else:
                prepared["sparse_embedding"] = {int(k): float(v) for k, v in sparse.items()}
 

    # Fill optional metadata fields
    for field, default_value in [
        ("column_headers", "[]"),
        ("table_units", "{}"),
        ("image_id", ""),
        ("image_path", ""),
        ("image_classification", ""),
    ]:
        if prepared.get(field) is None:
            prepared[field] = default_value


    prepared_chunks.append(prepared)


print(f"Prepared: {len(prepared_chunks)} / {len(all_chunks)} chunks")

In [ ]:
batch_size = 100
for i in range(0, len(prepared_chunks), batch_size):
    batch = prepared_chunks[i:i + batch_size]
    client.insert(collection_name=COLLECTION_NAME, data=batch)